# Datamine GETSAMP Process Example

This notebook demonstrates how to configure and run the **`getsamp`** process wrapper in `dmstudio`.

## Process Description

## GETSAMP

**GETSAMP** produces a sample file from a desurveyed drillhole file. In one way, it's almost the opposite of **[desurveying](<../COMMON/Drillhole%20Representation%20in%20Studio.md>)** component drillhole files into a **[static drillhole file](<../COMMON/Drillhole%20Representation%20in%20Studio.md>)**.

The output file will contain the BHID, FROM, TO from the input hole(s) with the longest possible intervals for each selected field.

A sample file may be used to save lithology or ore zone interpretations (for example, created with changes to the drillhole file imparted by **[group-lithology](<../command_help/group-lithology.md>)** , [assign-lithology](<../command_help/assign-lithology.md>) or [**edit-dh-attributes**](<../command_help/edit-dh-attributes.md>)).

It may also be used to save interpretations made to drillholes during **implicit modelling** in Studio's resource modelling products. During these activities, specific columns are added to drillholes to indicate enabling and reversing of columns (e.g. _FLAG_VS_ROCK or _FLAG_CS ).

The resulting downhole sample file can then be merged with existing drillhole sample tables during desurveying.

**Note** : data generated with absent values for all of the fields (F1, F2 etc) is automatically removed from the output of GETSAMP.

### Input Files:

* **in** (Static Drillhole File):
  Desurveyed sample data file. This will include fields BHID, FROM, TO, X, Y, Z, A0, B0,
  and all other fields which were included in the sample file(s). Expects fields BHID,
  XCOLLAR, YCOLLAR and ZCOLLAR; optional fields BRG, DIP.
  Required=Yes

### Output Files:

* **sample** (Table):
  Generated sample data file. This file is compulsory and will include fields BHID, FROM,
  and TO. It will also include at least one sample attribute field, such as grade or
  lithology.
  Required=Yes

### Fields:

* **f1** (Any : IN):
  First sample attribute field for saving to sample output.
  Default=Undefined
  Required=Yes

* **f2_f30** (Any : IN):
  Sample attribute fields for saving to sample output.
  Default=Undefined
  Required=No

### Parameters:

* **csvout**:
  Set to 1 to create a CSV output file of the plot data file specified in GETSAMP.
  Range=0-1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('getsamp')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute getsamp
print("Running getsamp...")
dm_cmd.getsamp(
    in_i='t_assays',  # required
    sample_o='t_getsamp_out',  # required
    f1_f='optional',  # required
    # f2_f30_f='optional',  # optional
    # csvout_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("getsamp execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_getsamp_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")